<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_8_%D0%90%D0%B2%D1%82%D0%BE%D0%BD%D0%BE%D0%BC%D0%BD%D1%8B%D0%B5_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%8B_%D0%BF%D0%BB%D0%B0%D0%BD%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B8_%D1%81%D0%B0%D0%BC%D0%BE%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.8. Автономные агенты: планирование и самооценка

## Введение: от реактивных помощников к проактивным агентам

Поздравляю! Мы прошли невероятный путь. В Лекции 6.1 мы создавали простого RAG-агента, который отвечал на вопросы по документам. В Лекции 6.4 мы научили агента вызывать инструменты и принимать решения. В Лекции 6.6 мы построили команду специализированных агентов, а в Лекции 6.7 добавили супервайзера, который управляет этой командой. Но все эти агенты были **реактивными** – они получали один вопрос, выполняли несколько шагов и останавливались. Они не планировали долгосрочные действия, не оценивали свой прогресс и не корректировали план на ходу.

Теперь мы подходим к **вершине** – автономным агентам. Представьте, что вы даёте агенту не вопрос, а **цель**: «Напиши аналитический отчёт о состоянии рынка ИИ в 2026 году». Агент сам:
1. Разбивает задачу на подзадачи (собрать данные, проанализировать, написать текст, отредактировать).
2. Выполняет их последовательно, используя доступные инструменты.
3. Проверяет качество своей работы.
4. Если результат неудовлетворительный – возвращается и исправляет ошибки.

Это и есть **автономный агент** – система, которая самостоятельно планирует, действует, оценивает и адаптируется до тех пор, пока цель не будет достигнута. Это уже не просто «ответ на вопрос», а настоящий цифровой помощник, способный решать сложные многошаговые задачи.

В этой лекции мы реализуем такого агента. Мы объединим три ключевых паттерна:
- **ReAct (Reasoning + Acting)** – агент чередует размышления и действия.
- **Plan‑and‑Execute** – агент сначала составляет план, потом выполняет.
- **Reflexion** – агент оценивает свои результаты и учится на ошибках.

К концу лекции вы получите полностью автономного агента, который сможет самостоятельно выполнять сложные задачи – от написания отчётов до проведения исследований. Поехали!

---

## Тема 1. Что такое автономный агент и зачем он нужен

### 1.1. Отличие от реактивного агента

Все агенты, которые мы строили ранее, были **реактивными**. Они получали запрос и выполняли заранее определённую последовательность действий. Даже супервайзер из Лекции 6.7, хотя и принимал решения на каждом шаге, делал это в рамках одной задачи. Как только ответ был сгенерирован – работа завершалась.

**Автономный агент** работает иначе:

| Аспект | Реактивный агент | Автономный агент |
|--------|------------------|------------------|
| **Инициатива** | Отвечает на запрос | Самостоятельно ставит подцели |
| **Планирование** | Фиксированный порядок шагов | Динамический план, который может меняться |
| **Оценка** | Не проверяет качество | Проверяет результат и исправляет ошибки |
| **Остановка** | Останавливается после ответа | Работает до достижения цели |
| **Адаптивность** | Не меняет стратегию | Меняет план при неудачах |

**Пример:** если реактивному агенту сказать «напиши отчёт», он может просто сгенерировать текст на основе имеющихся данных. Автономный агент сначала подумает: «Что нужно для отчёта? Какие данные собрать? Где их взять? Проверить ли факты?» – и только потом начнёт действовать, постоянно оценивая прогресс.

### 1.2. Примеры задач для автономных агентов

Автономные агенты особенно полезны там, где задача не может быть решена за один шаг:

- **Написание аналитического отчёта** – сбор данных, анализ, структурирование, написание, редактура.
- **Исследование темы** – поиск информации, проверка источников, синтез знаний, формулировка выводов.
- **Автоматизация рутины** – обработка писем, планирование встреч, управление проектами.
- **Обучение и саморазвитие** – агент изучает новую тему и проверяет свои знания.
- **Программирование** – написание кода, тестирование, отладка, рефакторинг.

В каждом из этих сценариев агенту нужно не просто ответить, а **достичь цели** – и для этого требуется планирование, оценка и адаптация.

### 1.3. Основные компоненты автономного агента

Автономный агент состоит из трёх ключевых компонентов, работающих в цикле:

1. **Планировщик (Planner)** – разбивает цель на подзадачи и определяет порядок их выполнения.
2. **Исполнитель (Executor)** – выполняет подзадачи, используя доступные инструменты.
3. **Оценщик (Evaluator / Self‑Critic)** – анализирует результат, проверяет, достигнута ли цель, и решает, нужно ли корректировать план.

К этим трём добавляется **память** – не только краткосрочная (история диалога), но и долгосрочная (сохранение результатов предыдущих шагов, чтобы не повторять их).

**Схема работы:**

```
Пользователь задаёт цель
        ↓
┌───────────────────────────────────────┐
│  Планировщик (LLM)                    │
│  "Что нужно сделать для достижения?   │
│   Какой следующий шаг?"               │
└──────────────────┬────────────────────┘
                   ↓
┌───────────────────────────────────────┐
│  Исполнитель (Executor)               │
│  Выполняет шаг (вызов инструмента,    │
│  поиск, вычисление)                   │
└──────────────────┬────────────────────┘
                   ↓
┌───────────────────────────────────────┐
│  Оценщик (Self‑Critic)                │
│  "Достигнута ли цель? Нужно ли        │
│   изменить план?"                     │
└──────────────────┬────────────────────┘
                   ↓
        ┌──────────┴──────────┐
        │  Цель достигнута?    │
        │  Да → Ответ          │
        │  Нет → Вернуться к   │
        │        планировщику  │
        └─────────────────────┘
```

Каждый из этих компонентов может быть реализован как отдельный LLM-агент или как один агент, который выполняет все три роли в цикле.

### 1.4. Обзор подходов к автономности

Существует несколько популярных паттернов для построения автономных агентов:

#### ReAct (Reasoning + Acting)

Агент чередует «размышление» и «действие». На каждом шаге он пишет, что собирается сделать, выполняет действие, анализирует результат и решает, что делать дальше.

```
Шаг 1: "Мне нужно найти информацию о компании X" → поиск
Шаг 2: "Теперь нужно сравнить с компанией Y" → поиск
Шаг 3: "Данные собраны, можно писать ответ" → генерация
```

Этот паттерн мы уже использовали в Лекции 6.4 – агент с инструментами по сути работал по схеме ReAct.

#### Plan‑and‑Execute

Агент сначала составляет полный план действий, а затем последовательно его выполняет. Это делает поведение более предсказуемым и позволяет видеть весь маршрут заранее.

```
План:
1. Найти информацию о компании X
2. Найти информацию о компании Y
3. Сравнить показатели
4. Написать отчёт
5. Проверить факты
→ Выполнение по шагам
```

#### Reflexion (Самооценка)

Агент генерирует ответ, затем критикует его, указывает на ошибки и генерирует исправленный вариант. Этот цикл повторяется, пока качество не станет удовлетворительным.

```
Шаг 1: "Вот мой ответ..." (генерация)
Шаг 2: "Я ошибся в датах, нужно исправить" (рефлексия)
Шаг 3: "Вот исправленный ответ" (новая генерация)
```

#### Tree of Thoughts (Дерево мыслей)

Агент генерирует несколько вариантов решения, оценивает каждый и выбирает лучший. Это похоже на то, как человек рассматривает разные варианты перед принятием решения.

В этой лекции мы объединим **Plan‑and‑Execute** и **Reflexion**, чтобы получить максимально автономного агента.

### 1.5. Какие пакеты нужны

Для реализации автономного агента нам не понадобятся новые библиотеки. Всё, что мы использовали раньше, остаётся:

```bash
pip install langchain langchain-ollama langgraph chromadb sentence-transformers
```

Мы будем использовать:
- **LangGraph** – для построения графа с циклом (планировщик → исполнитель → оценщик → планировщик).
- **LangChain** – для работы с LLM и инструментами.
- **Ollama** – как локальный сервер для LLM.
- **Chroma** – для долгосрочной памяти (сохранение результатов шагов).

Никаких дополнительных установок не требуется – всё уже есть в вашем проекте.

---

## Краткий итог Тема 1

- **Автономный агент** – это система, которая самостоятельно планирует, действует, оценивает и адаптируется до достижения цели.
- Он отличается от реактивного агента **проактивностью** – он не просто отвечает, а сам ставит подцели и корректирует стратегию.
- **Три ключевых компонента**: планировщик, исполнитель, оценщик.
- **Основные паттерны**: ReAct, Plan‑and‑Execute, Reflexion, Tree of Thoughts.
- Мы будем использовать **LangGraph** для реализации цикла и **Ollama** для LLM.

---

**В следующей теме мы перейдём к реализации планировщика и настроим цикл «план → действие → оценка».**


## Тема 2. Паттерн ReAct (Reasoning + Acting) (скрипт `react_agent.py`)

В предыдущей лекции мы говорили о том, что автономный агент должен уметь планировать, действовать и оценивать результат. Самый фундаментальный паттерн, на котором строится большинство автономных агентов, – это **ReAct (Reasoning + Acting)**. Мы уже неявно использовали его в Лекции 6.4, когда агент с инструментами решал, что вызывать, и выполнял действия. Но тогда мы не выделяли «мысли» как отдельный элемент. Теперь мы сделаем это осознанно и структурированно, добавив реальные инструменты и улучшенную наблюдаемость.

---

### 2.1. Идея: чередовать «подумать» и «сделать»

ReAct предлагает простую, но мощную идею: агент на каждом шаге сначала **думает** (рассуждает), затем **действует**, а потом **наблюдает** результат. Модель явно пишет свои мысли, что делает процесс **прозрачным** и **отлаживаемым**.

**Цикл ReAct:**

```
Пользователь: "Напиши отчёт о компании X"
Агент: Мысль: нужно найти информацию о компании X.
       Действие: search_docs("X")
Наблюдение: [результаты поиска]
Агент: Мысль: теперь нужно собрать финансовые показатели.
       Действие: search_docs("финансовые показатели X")
Наблюдение: [результаты поиска]
Агент: Мысль: данных достаточно, можно написать ответ.
       (нет действия, завершаем)
```

**Преимущества:**

- **Прозрачность** – мы видим, почему агент делает то или иное действие.
- **Отладка** – легко понять, на каком шаге произошла ошибка.
- **Гибкость** – агент может передумать, если наблюдение не совпадает с ожиданиями.

---

### 2.2. Формат промпта

Для реализации ReAct мы будем использовать следующий формат промпта:

```
Мысль: {мысль агента}
Действие: {имя инструмента, параметры}
Наблюдение: {результат выполнения инструмента}
Мысль: {следующая мысль}
...
```

В нашей реализации мы используем `bind_tools` для вызова инструментов и храним наблюдения в отдельном поле состояния, что позволяет агенту анализировать предыдущие шаги.

**Системный промпт для ReAct:**

```python
SYSTEM_PROMPT = """
Ты — автономный агент, работающий по методологии ReAct (Reasoning + Acting).

Твоя задача — достичь цели пользователя, используя инструменты. Всегда следуй этому циклу:

1. **Мысль** (Reasoning) – проанализируй, что известно, и что нужно сделать дальше.
2. **Действие** (Acting) – если нужна дополнительная информация, **вызови подходящий инструмент** (search_docs, web_search или calculate). Никогда не пиши действие текстом — всегда используй вызов инструмента.
3. **Наблюдение** (Observation) – после получения результата инструмента, **обязательно** проанализируй его текстом. Скажи, что ты узнал и достаточно ли этого для ответа.
4. Если цель ещё не достигнута, вернись к шагу 1 (новая Мысль).
5. Если цель достигнута, напиши **финальный ответ** пользователю (без вызова инструментов).

Важно:
- Не завершай работу, пока не будет достаточно информации для полного и точного ответа.
- Если результат инструмента пуст или не содержит нужных данных, попробуй другой инструмент или переформулируй запрос.
- В финальном ответе суммируй все наблюдения и дай чёткий, структурированный ответ.
"""
```

---

### 2.3. Реализация на LangGraph

Создадим граф с двумя узлами:

1. **`agent`** – вызывает LLM с инструментами, получает мысль и действие.
2. **`tools`** – выполняет действие, возвращает наблюдение.

**Цикл:**

```
agent → tools → agent → tools → ... → завершение
```

**Состояние** содержит:
- `messages` – история сообщений (включая мысли, действия, наблюдения).
- `iteration` – счётчик шагов.
- `question` – исходная цель.
- `observations` – список наблюдений для анализа.

---

### 2.4. Инструменты: реальный поиск и вычисления

В нашей реализации мы используем три инструмента:

1. **`search_docs`** – поиск в локальной базе знаний (заглушка для демонстрации).
2. **`web_search`** – реальный поиск через DuckDuckGo (без API-ключа).
3. **`calculate`** – безопасное выполнение математических вычислений.

```python
from langchain_community.tools import DuckDuckGoSearchRun

web_search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете с помощью DuckDuckGo."""
    try:
        results = web_search_tool.invoke(query)
        return results[:1000] if len(results) > 1000 else results
    except Exception as e:
        return f"Ошибка поиска: {e}"
```

---

### 2.5. Полный код `react_agent.py`

Ниже представлен полный код агента с реализацией паттерна ReAct. Он включает реальный веб-поиск, хранение наблюдений, улучшенный системный промпт и защиту от бесконечного цикла.

```python
"""
react_agent.py - Реализация паттерна ReAct на LangGraph (улучшенная версия)
Лекция 6.8, Тема 2

Особенности:
- Реальный веб-поиск через DuckDuckGo
- Хранение наблюдений для анализа
- Улучшенный системный промпт
- Защита от бесконечного цикла
"""

import math
import re
from typing import TypedDict, List, Annotated, Literal, Optional
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun  # реальный поиск

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def search_docs(query: str) -> str:
    """Ищет информацию в локальной базе знаний."""
    # Для демонстрации оставляем заглушку, но можно заменить на реальный ретривер
    if "RAG" in query.upper():
        return "RAG (Retrieval-Augmented Generation) — это подход, сочетающий поиск и генерацию. Основные компоненты: ретривер (Chroma, FAISS) и генератор (LLM)."
    elif "LLM" in query.upper():
        return "Большие языковые модели (LLM) обучаются на больших объёмах текстов."
    else:
        return "Информация не найдена."

# Реальный веб-поиск (без API-ключа)
web_search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Выполняет поиск в интернете с помощью DuckDuckGo."""
    try:
        results = web_search_tool.invoke(query)
        # Ограничим длину, чтобы не перегружать контекст
        return results[:1000] if len(results) > 1000 else results
    except Exception as e:
        return f"Ошибка поиска: {e}"

@tool
def calculate(expression: str) -> str:
    """Выполняет математические вычисления (безопасно)."""
    # Разрешаем только числа и простые операторы
    if not re.match(r'^[\d+\-*/().\s]+$', expression):
        return "Ошибка: недопустимые символы в выражении."
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

tools = [search_docs, web_search, calculate]

# ============================================================================
# 2. НАСТРОЙКА LLM
# ============================================================================

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)
llm_with_tools = llm.bind_tools(tools)

# Улучшенный системный промпт
SYSTEM_PROMPT = """
Ты — автономный агент, работающий по методологии ReAct (Reasoning + Acting).

Твоя задача — достичь цели пользователя, используя инструменты. Всегда следуй этому циклу:

1. **Мысль** (Reasoning) – проанализируй, что известно, и что нужно сделать дальше.
2. **Действие** (Acting) – если нужна дополнительная информация, **вызови подходящий инструмент** (search_docs, web_search или calculate). Никогда не пиши действие текстом — всегда используй вызов инструмента.
3. **Наблюдение** (Observation) – после получения результата инструмента, **обязательно** проанализируй его текстом. Скажи, что ты узнал и достаточно ли этого для ответа.
4. Если цель ещё не достигнута, вернись к шагу 1 (новая Мысль).
5. Если цель достигнута, напиши **финальный ответ** пользователю (без вызова инструментов).

Важно:
- Не завершай работу, пока не будет достаточно информации для полного и точного ответа.
- Если результат инструмента пуст или не содержит нужных данных, попробуй другой инструмент или переформулируй запрос.
- В финальном ответе суммируй все наблюдения и дай чёткий, структурированный ответ.

Инструменты:
- search_docs(query) – поиск в локальной базе знаний (ограниченная информация).
- web_search(query) – поиск в интернете (актуальные данные).
- calculate(expression) – вычисление математических выражений.

Начинай!
"""

# ============================================================================
# 3. СОСТОЯНИЕ (добавили observations)
# ============================================================================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    question: str
    iteration: int
    max_iterations: int
    observations: List[str]   # храним результаты наблюдений

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def agent_node(state: AgentState) -> dict:
    iteration = state.get("iteration", 0) + 1
    print(f"\n🧠 Итерация {iteration}")

    if iteration > state.get("max_iterations", 10):
        print("⚠️  Превышен лимит итераций.")
        return {
            "messages": [AIMessage(content="Не удалось достичь цели за отведённое время.")],
            "iteration": iteration
        }

    messages = state["messages"]
    # Добавляем системный промпт, если его нет
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    # Вставляем напоминание о наблюдениях в контекст, если они есть
    observations = state.get("observations", [])
    if observations:
        obs_text = "\n".join([f"Наблюдение {i+1}: {obs}" for i, obs in enumerate(observations)])
        reminder = f"\n\nТвои предыдущие наблюдения:\n{obs_text}\n\nТеперь, основываясь на них, реши, что делать дальше."
        messages.append(HumanMessage(content=reminder))

    response = llm_with_tools.invoke(messages)
    print(f"💬 Ответ LLM: {response.content[:150]}..." if response.content else "💬 Ответ LLM: (пусто)")

    if hasattr(response, "tool_calls") and response.tool_calls:
        print(f"🔧 Вызваны инструменты: {[tc['name'] for tc in response.tool_calls]}")
    else:
        print("✅ Инструменты не вызваны, возможно финальный ответ.")

    return {"messages": [response], "iteration": iteration}

def tools_node(state: AgentState) -> dict:
    last_message = state["messages"][-1]
    tool_calls = last_message.tool_calls

    if not tool_calls:
        return {"messages": []}

    tool_messages = []
    observations = state.get("observations", [])

    for tc in tool_calls:
        tool_name = tc["name"]
        tool_args = tc["args"]
        tool_map = {t.name: t for t in tools}
        if tool_name in tool_map:
            result = tool_map[tool_name].invoke(tool_args)
            print(f"🔧 Инструмент {tool_name} вернул: {result[:100]}...")
            # Сохраняем наблюдение
            observations.append(f"{tool_name}({tool_args}) -> {result}")
            tool_messages.append(
                ToolMessage(content=str(result), tool_call_id=tc["id"])
            )
        else:
            err_msg = f"Инструмент {tool_name} не найден."
            observations.append(err_msg)
            tool_messages.append(
                ToolMessage(content=err_msg, tool_call_id=tc["id"])
            )

    return {"messages": tool_messages, "observations": observations}

# ============================================================================
# 5. МАРШРУТИЗАЦИЯ
# ============================================================================

def route_after_agent(state: AgentState) -> Literal["tools", "finish"]:
    last_message = state["messages"][-1]
    # Если есть вызовы инструментов — идём в tools
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    # Если инструменты не вызваны, считаем, что это финальный ответ
    return "finish"

# ============================================================================
# 6. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)

builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {
        "tools": "tools",
        "finish": END
    }
)
builder.add_edge("tools", "agent")

graph = builder.compile()

# ============================================================================
# 7. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    questions = [
        "Что такое RAG и как он работает?",
        "Сколько будет 25% от 200?",
        "Сравни RAG и обычный ChatGPT.",
        "Какая сегодня погода в Москве?"  # проверим реальный поиск
    ]

    for q in questions:
        print("\n" + "=" * 60)
        print(f"📝 Вопрос: {q}")
        print("=" * 60)

        initial_state = {
            "messages": [HumanMessage(content=q)],
            "question": q,
            "iteration": 0,
            "max_iterations": 10,   # увеличено
            "observations": []
        }

        try:
            result = graph.invoke(initial_state, config={"recursion_limit": 15})
        except Exception as e:
            print(f"❌ Ошибка выполнения: {e}")
            continue

        # Извлекаем финальный ответ (последнее AIMessage без tool_calls)
        final_answer = None
        for msg in reversed(result["messages"]):
            if isinstance(msg, AIMessage) and not (hasattr(msg, "tool_calls") and msg.tool_calls):
                final_answer = msg.content
                break

        print(f"\n✅ Финальный ответ:\n{final_answer if final_answer else 'Не сгенерирован'}")
        print(f"📊 Шагов выполнено: {result.get('iteration', 0)}")
        print(f"📋 Наблюдений: {len(result.get('observations', []))}")
        for i, obs in enumerate(result.get('observations', []), 1):
            print(f"   {i}. {obs[:150]}...")
```

---

### 2.6. Результаты тестирования

При запуске на четырёх вопросах агент показал:

| Вопрос | Действие | Результат |
|--------|----------|-----------|
| «Что такое RAG?» | 2 поиска, затем ответ | Агент собрал информацию и сгенерировал ответ |
| «25% от 200?» | 1 вычисление, затем ответ | Агент правильно вычислил и ответил |
| «Сравни RAG и ChatGPT» | 2 поиска, остановился на мысли | Требует доработки (Reflexion) |
| «Погода в Москве» | 2 одинаковых поиска, остановился | Требует доработки (Reflexion) |

**Что работает хорошо:**
- Агент правильно выбирает инструменты.
- Сохраняет наблюдения и использует их.
- Останавливается при достижении цели.

**Что требует улучшения:**
- Сложные вопросы требуют самооценки (Reflexion).
- Агент иногда повторяет одинаковые действия.
- Не всегда чётко отделяет финальный ответ от мыслей.

Эти проблемы будут решены в следующей теме – **Reflexion (самооценка и рефлексия)**.

---

## Краткий итог Тема 2

- **ReAct** – фундаментальный паттерн, где агент чередует «мысли» и «действия».
- Мы реализовали агента с реальным веб-поиском через DuckDuckGo.
- Агент сохраняет наблюдения и использует их для принятия решений.
- Цикл продолжается до тех пор, пока агент не перестанет вызывать инструменты или не достигнет лимита итераций.
- Для простых задач агент работает отлично, для сложных требуется самооценка.

---

**В следующей теме мы добавим самооценку и рефлексию – научим агента проверять свои ответы и исправлять ошибки.**